In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import h5py
import scipy.sparse as sp
import yaml
import time
import gget
from scipy.stats import zscore

import torch

import anndata as an
import scanpy as sc
import rapids_singlecell as rsc
import scvi

from scvi.external import CellAssign

import cupy as cp
from cuml.manifold import TSNE
from cuml.decomposition import PCA

sc.settings.verbosity = 3

# Define Marker Genes from DEG

In [ ]:
%%time
fpath = "/scratch/indikar_root/indikar1/shared_data/hematokytos/sample_1_adata_basename_deg.parquet"

df = pd.read_parquet(fpath)
print(f"Raw data shape: {df.shape=}")
df = df.sort_values(by=['group', 'logfoldchanges'], ascending=[True, False])

print(df.head().to_string())

In [ ]:
# read tf list
fpath = "../../resources/allTFs_hg38.txt"
tf_list = [x.strip() for x in open(fpath)]
print(f"{len(tf_list)=}")
tf_list[:10]

In [ ]:
# Define thresholds
min_logfc = 1.0
max_padj = 0.05
min_pct_group = 0.35
# max_pct_ref = 0.25

# Apply thresholds
marker_df = df[
    # (df['names'].isin(tf_list)) &
    (df['logfoldchanges'] > min_logfc) &
    (df['pvals_adj'] < max_padj) &
    (df['pct_nz_group'] > min_pct_group) 
    # (df['pct_nz_reference'] < max_pct_ref)
]

# Optional: sort by score or logFC within each group
marker_df = marker_df.sort_values(['group', 'logfoldchanges'], ascending=[True, False])
marker_df['is_marker'] = 1
print(f"{marker_df.shape=}")
marker_df.head()

In [ ]:
top_n_genes = 100

top_markers = (
    marker_df
    .groupby('group', group_keys=False)
    .apply(lambda g: g.nlargest(top_n_genes, 'scores'))
)

print(top_markers[['group', 'names', 'logfoldchanges', 'pct_nz_group', 'pct_nz_reference']].head().to_string(index=False))

In [ ]:
# Pivot into wide format: rows=genes, columns=groups, values=1 if marker
marker_mat = top_markers.pivot_table(
    index='names',
    columns='group',
    values='is_marker',
    fill_value=0
)
print(f"{marker_mat.shape=}")
marker_mat.head()

# Load the data

In [ ]:
%%time

start = time.perf_counter()

fpath = "/scratch/indikar_root/indikar1/shared_data/hematokytos/processed/sample_1_adata.h5ad"
raw_adata = sc.read_h5ad(fpath)
print(f"Raw: {raw_adata.shape=}")

raw_adata.var_names_make_unique()
raw_adata.obs_names_make_unique()

# add library size
lib_size = raw_adata.X.sum(1)
raw_adata.obs["size_factor"] = lib_size / np.mean(lib_size)

sc.logging.print_memory_usage() # Log memory at the end

print(f"Total time: {time.perf_counter() - start:.2f} seconds")

raw_adata # Return/output adata

# Sampling

In [ ]:
fraction = 0.15
adata = raw_adata.copy()
adata = adata[:, marker_mat.index].copy()
adata = sc.pp.subsample(adata, fraction=fraction, copy=True)
print(f"Filtered: {raw_adata.shape=}")
adata

In [ ]:
adata.obs_names[:10]

In [ ]:
# break

# Training

In [ ]:
print(torch.cuda.memory_allocated() / 1e6, "MB allocated")
print(torch.cuda.memory_reserved() / 1e6, "MB reserved")

In [ ]:
%%time
torch.cuda.empty_cache() 
torch.cuda.ipc_collect() 

scvi.external.CellAssign.setup_anndata(
    adata, 
    size_factor_key="size_factor",
    layer='counts',
)

model = CellAssign(adata, marker_mat)
model.train(
    max_epochs=200,
    lr=0.001,
    accelerator='gpu',
    shuffle_set_split=False,
)

model.history["elbo_validation"].plot()

In [ ]:
preds = model.predict()
print(f"{preds.shape=}")
print(preds.head().to_string())

In [ ]:
sns.clustermap(preds.sample(35), cmap="viridis")

In [ ]:
# Get counts
pred_counts = preds.idxmax(axis=1).value_counts().rename("predicted")
true_counts = adata.obs['basename'].value_counts().rename("true")

# Combine counts
df_counts = pd.concat([true_counts, pred_counts], axis=1).fillna(0).astype(int)

# Add difference (count)
df_counts["difference"] = df_counts["predicted"] - df_counts["true"]

# Add percentages
df_counts["true_pct"] = df_counts["true"] / df_counts["true"].sum() * 100
df_counts["predicted_pct"] = df_counts["predicted"] / df_counts["predicted"].sum() * 100
df_counts["pct_diff"] = df_counts["predicted_pct"] - df_counts["true_pct"]

# Optional: sort
df_counts["abs_pct_diff"] = df_counts["pct_diff"].abs()
df_counts = df_counts.sort_values("abs_pct_diff", ascending=False).drop(columns="abs_pct_diff")

# Format percentages to 1 decimal place
df_counts["true_pct"] = df_counts["true_pct"].round(1)
df_counts["predicted_pct"] = df_counts["predicted_pct"].round(1)
df_counts["pct_diff"] = df_counts["pct_diff"].round(1)

print(df_counts.to_string())

In [ ]:
confusion_matrix = pd.crosstab(
    preds.idxmax(axis=1),
    adata.obs['basename'].values,
    rownames=["y Pred"],
    colnames=["y True"],
)

confusion_matrix /= confusion_matrix.sum(1).ravel().reshape(-1, 1)
# Plot
fig, ax = plt.subplots(figsize=(9, 9))  # Square aspect
sns.heatmap(
    confusion_matrix,
    ax=ax,
    square=True,
    cmap="Blues",  # Use "viridis" or "mako" for alternatives
    cbar_kws={"label": "proportion", "shrink": 0.4},
    annot=True,
    fmt=".2f",
    linewidths=0.5,
    linecolor='k'
)

# Aesthetics
ax.set_xlabel("True Label", fontsize=12)
ax.set_ylabel("Predicted Label", fontsize=12)
plt.xticks(rotation=90, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
plt.rcParams['figure.dpi'] = 200
plt.rcParams['figure.figsize'] = 4.25, 4.75

adata.obs['prediction'] = preds.idxmax(axis=1).values

sc.pl.tsne(
    adata,
    color=['basename', 'prediction'],
    size=35,
    ncols=4,
    alpha=1,
    use_raw=False,
    add_outline=True,
    outline_color=('k', 'k'),
    frameon=False,
    colorbar_loc=None,
    wspace=0.75,
    palette=adata.uns['basename_palette'],
)

In [ ]:
plt.rcParams['figure.dpi'] = 200
plt.rcParams['figure.figsize'] = 4.75, 4.75

adata.obs['p(x)'] = preds.max(axis=1).fillna(0.0).values
min_val = adata.obs['p(x)'].min()
max_val = adata.obs['p(x)'].max()
adata.obs['p(x)_reversed_norm'] = (max_val - adata.obs['p(x)']) / (max_val - min_val)

sc.pl.tsne(
    adata,
    color=['p(x)_reversed_norm'],
    sort_order=True,
    size=32,
    ncols=4,
    alpha=0.75,
    use_raw=False,
    add_outline=True,
    outline_color=('k', 'k'),
    frameon=False,
    colorbar_loc=None,
    wspace=0.75,
    title=['uncertainty'],
    cmap='inferno',
)

# with pangloa

In [ ]:
def load_pathway(fpath):
    """
    Loads an Enrichr-like database file into a boolean DataFrame.

    Args:
        fpath (str): Path to the Enrichr-like database file.

    Returns:
        pandas.DataFrame: A boolean DataFrame where:
            - Index: Genes
            - Columns: Pathways
            - Values: True if the gene is in the pathway, False otherwise.
    """

    result = []
    with open(fpath,  encoding='utf-8') as f:
        for line in f:
            split_line = [x for x in line.strip().split('\t') if x]  # Remove empty strings directly

            row = {'label': split_line[0]}
            for gene in split_line[1:]:
                row[gene] = 1

            result.append(row)

    df = pd.DataFrame(result)
    df = df.fillna(0.0).set_index('label').astype(bool).T  # Chained operations for clarity

    return df

fpath = "../../resources/PanglaoDB_Augmented_2021.txt"
pdf = load_pathway(fpath)
cell_types = ['Hematopoietic Stem Cells', 'Endothelial Cells', 'Fibroblasts']

pdf = pdf[pdf[cell_types].any(axis=1)][cell_types]
print(f"Raw: {pdf.shape=}")
pdf = pdf.astype(int)
pdf = pdf[pdf.index.isin(raw_adata.var_names)]
print(f"Filtered: {pdf.shape=}")
pdf.head()

# Sampling

In [ ]:
fraction = 0.15
adata = raw_adata.copy()
adata = adata[:, pdf.index].copy()
adata = sc.pp.subsample(adata, fraction=fraction, copy=True)
print(f"Filtered: {raw_adata.shape=}")

adata

In [ ]:
%%time
torch.cuda.empty_cache() 
torch.cuda.ipc_collect() 

scvi.external.CellAssign.setup_anndata(
    adata, 
    size_factor_key="size_factor",
    layer='counts',
)

model = CellAssign(adata, marker_mat)
model.train(
    max_epochs=200,
    lr=0.001,
    accelerator='gpu',
    shuffle_set_split=False,
)

model.history["elbo_validation"].plot()

In [ ]:
preds = model.predict()
print(f"{preds.shape=}")
print(preds.head().to_string())

In [ ]:
confusion_matrix = pd.crosstab(
    preds.idxmax(axis=1),
    adata.obs['basename'].values,
    rownames=["y Pred"],
    colnames=["y True"],
)

confusion_matrix /= confusion_matrix.sum(1).ravel().reshape(-1, 1)
# Plot
fig, ax = plt.subplots(figsize=(9, 9))  # Square aspect
sns.heatmap(
    confusion_matrix,
    ax=ax,
    square=True,
    cmap="Blues",  # Use "viridis" or "mako" for alternatives
    cbar_kws={"label": "proportion", "shrink": 0.4},
    annot=True,
    fmt=".2f",
    linewidths=0.5,
    linecolor='k'
)

# Aesthetics
ax.set_xlabel("True Label", fontsize=12)
ax.set_ylabel("Predicted Label", fontsize=12)
plt.xticks(rotation=90, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
plt.rcParams['figure.dpi'] = 200
plt.rcParams['figure.figsize'] = 4.25, 4.75

adata.obs['prediction'] = preds.idxmax(axis=1).values

sc.pl.tsne(
    adata,
    color=['basename', 'prediction'],
    size=35,
    ncols=4,
    alpha=1,
    use_raw=False,
    add_outline=True,
    outline_color=('k', 'k'),
    frameon=False,
    colorbar_loc=None,
    wspace=0.75,
    palette=adata.uns['basename_palette'],
)